In [1]:
import pandas as pd

df_final = pd.read_csv('df_final.csv')
df_final

,Unnamed: 0,"Side (1-Right, 2-Left)",Subject ID,"Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)",Knee Injury History,"Sex (0-Female, 1-Male)",Age at testing [years],Time since injury [months],Height [cm],Weight [kg],Contralateral Knee Injury,VM_at_80,VM_Mean,VL_Mean,VM_VL_Ratio
0,0,1,12,0,1,1,30,144.5,168.8,69.0,1,0.037942,0.0100,0.0174,0.574713
1,1,2,12,1,1,1,30,144.5,168.8,69.0,1,0.053714,0.0109,0.0173,0.630058
2,2,1,99,1,1,0,26,117.0,167.2,61.2,1,0.026152,0.0075,0.0148,0.506757
3,3,2,99,0,1,0,26,117.0,167.2,61.2,1,0.027840,0.0089,0.0173,0.514451
4,4,1,112,0,1,0,27,93.6,170.0,58.7,0,0.039857,0.0098,0.0157,0.624204
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,117,2,254,1,0,1,18,999.0,174.0,61.9,0,0.043378,0.0117,0.0181,0.646409
118,118,1,255,1,1,1,19,38.0,195.0,102.3,0,0.032077,0.0110,0.0171,0.643275
119,119,2,255,0,1,1,19,38.0,195.0,102.3,0,0.052564,0.0110,0.0170,0.647059
120,120,1,256,0,0,1,23,999.0,187.5,80.0,0,0.033372,0.0125,0.0212,0.589623


In [2]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. 현경님이 알려주신 피지오넷 데이터 경로
physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

try:
    # 피지오넷 폴더 내의 CSV 파일 중 하나를 샘플로 읽어옵니다 (보통 복수 파일일 수 있음)
    # 여기서는 현경님이 분석하셨던 df_31의 원천 파일을 찾는 로직입니다.
    csv_files = [f for f in os.listdir(physio_folder) if f.endswith('.csv')]
    
    if len(csv_files) > 0:
        # 첫 번째 파일을 대조군으로 사용 (필요시 파일명 특정 가능)
        df_p = pd.read_csv(os.path.join(physio_folder, csv_files[0]))
        print(f"✅ 피지오넷 데이터 로드 성공! 파일명: {csv_files[0]}")

        # 2. 학습 데이터 구성 (df_final은 이전 셀에서 이미 로드됨)
        # 부상자 그룹 (Target: 1)
        df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
        df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
        df_target_1['Target'] = 1

        # 정상군 그룹 (Target: 0)
        # 피지오넷 데이터의 LAT.V를 비교군으로 사용
        df_target_0 = pd.DataFrame({
            'Age': [24] * len(df_p), 
            'Sex': [0] * len(df_p),
            'Muscle_Val': df_p['LAT.V'], 
            'Target': 0
        })

        # 3. 데이터 합체 및 머신러닝
        df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)
        X = df_ml[['Age', 'Sex', 'Muscle_Val']]
        y = df_ml['Target']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_model.fit(X_train, y_train)

        # 4. 최종 결과 출력
        y_pred = rf_model.predict(X_test)
        print(f"\n🎯 무릎 부상 예측 모델 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
        print("-" * 40)
        print(classification_report(y_test, y_pred))
        
        # 기여도 확인
        for name, val in zip(X.columns, rf_model.feature_importances_):
            print(f"🔍 {name}의 예측 기여도: {val:.4f}")
    else:
        print("❌ 폴더 내에 CSV 파일이 보이지 않아요! 확장자를 확인해 주세요.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

✅ 피지오넷 데이터 로드 성공! 파일명: Subject_info.csv
❌ 에러 발생: 'LAT.V'


In [4]:
df_p

,N.;Signal name;Gender;Age (years);Height (cm);Weight (Kg)
0,1;Subject1;F;25;152;45
1,2;Subject2;M;24;182;70
2,3;Subject3;F;24;158;49
3,4;Subject4;M;24;175;76
4,5;Subject5;F;22;164;50
5,6;Subject6;F;23;163;53
6,7;Subject7;M;27;181;66
7,8;Subject8;F;23;164;50
8,9;Subject9;M;25;181;63
9,10;Subject10;F;24;165;60


In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. 부상자 데이터 (122명) 정리
# df_final은 아까 나이, 성별, VM_at_80 등이 합쳐진 표입니다.
df_injury = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80', 'VM_VL_Ratio']].copy()
df_injury['Target'] = 1  # 부상 이력 있음

# 2. 피지오넷 데이터 (31명) 정리 
# (아까 찾은 피지오넷 변수 이름을 df_p라고 가정합니다. 이름이 다르면 수정!)
df_normal = df_p[['REC.F', 'HAM', 'LAT.V', 'REC_HAM_Ratio']].copy()
# 피지오넷에도 나이/성별 데이터가 있다면 합쳐주세요. 없다면 일단 근육 지표 위주로!
df_normal['Target'] = 0  # 정상군

# 3. 두 데이터의 공통 분모 찾기 (Feature Selection)
# 모델이 학습할 공통 항목들만 추려냅니다. 
# 여기서는 예시로 VM(내측)과 관련 지표들을 사용합니다.
# (현경님의 데이터 상황에 따라 분석할 컬럼을 리스트업하세요)
features = ['VM_at_80', 'VM_VL_Ratio'] # 부상자 데이터용
features_p = ['LAT.V', 'REC_HAM_Ratio'] # 피지오넷 데이터용 (대응되는 지표들)

# 4. 최종 머신러닝용 데이터셋 만들기 (통합)
# 주의: 두 데이터의 컬럼 이름이 같아야 합쳐집니다.
df_injury.columns = ['Age', 'Sex', 'Muscle_Val', 'Ratio', 'Target']
# 피지오넷 데이터도 구조를 맞춰서 합쳐줍니다 (예시)
# df_normal_reshaped = ... 

print("✅ 데이터 통합 준비 완료! 이제 'df_total'로 머신러닝을 돌릴 수 있습니다.")

KeyError: "None of [Index(['REC.F', 'HAM', 'LAT.V', 'REC_HAM_Ratio'], dtype='object')] are in the [columns]"

In [ ]:
import pandas as pd
import numpy as np

# 1. 부상자 데이터(122명) 다시 만들기 (df_final 재정의)
# df_info와 df_individual이 메모리에 있다면 바로 합쳐집니다.
try:
    df_final = df_info.iloc[:122].copy()
    
    # 아까 만든 80포인트 값 다시 넣기
    target_point = 80
    m_idx = 1  # VM (내측광근)
    df_final['VM_at_80'] = M_array[:, m_idx * 250 + target_point]
    
    # 부상자 그룹 특징 추출 (나이, 성별, 근육값)
    df_injury = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_injury.columns = ['Age', 'Sex', 'Muscle_Val']
    df_injury['Target'] = 1  # 1은 부상군
    
    # 2. 피지오넷 데이터(31명) 정리 
    # (아까 찾은 피지오넷 변수 이름을 자동으로 찾아 합칩니다)
    found_p = None
    for name, obj in list(globals().items()):
        if isinstance(obj, pd.DataFrame) and 'TIB.A' in obj.columns:
            found_p = obj.copy()
            break
            
    if found_p is not None:
        # 피지오넷은 'LAT.V'를 대표 근육값으로 사용 (부상자의 VM과 대응)
        df_normal = pd.DataFrame()
        # 피지오넷에 나이가 없다면 평균값(예: 24세)으로 채우거나 0으로 둡니다.
        df_normal['Age'] = [24] * len(found_p) # 임시값, 실제 데이터 있다면 수정!
        df_normal['Sex'] = [0] * len(found_p)  # 임시값
        df_normal['Muscle_Val'] = found_p['LAT.V']
        df_normal['Target'] = 0  # 0은 정상군
        
        # 3. 드디어 두 데이터 합체!
        df_ml = pd.concat([df_injury, df_normal], axis=0).reset_index(drop=True)
        print(f"✅ 통합 완료! 전체 데이터 수: {len(df_ml)}행")
        print(df_ml.head())
    else:
        print("❌ 피지오넷 데이터를 찾을 수 없습니다. 피지오넷 로드 셀을 다시 실행해주세요.")

except NameError as e:
    print(f"❌ 아직 변수가 부족합니다: {e}")
    print("df_info나 M_array가 정의된 셀들을 위에서부터 다시 실행해주셔야 해요!")

In [ ]:
import pandas as pd
import scipy.io
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. 파일 로드 (M.mat에서 데이터 추출)
mat_data = scipy.io.loadmat('M.mat')
# mat 파일 안에 'M'이라는 이름으로 저장되어 있다고 가정합니다.
M_data = mat_data['M'] 

# 2. 엑셀에서 환자 정보 로드 (df_info)
df_info = pd.read_excel('Anonymized_Data_Identifiers.xlsx')

# 3. 머신러닝용 데이터 정리
# 부상자 그룹 (122명) 데이터 추출
df_ml_data = pd.DataFrame({
    'Age': df_info['Age at testing [years]'].iloc[:122],
    'Sex': df_info['Sex (0-Female, 1-Male)'].iloc[:122],
    # VM(내측광근)의 80포인트 지점 데이터 추출
    'VM_at_80': M_data[:122, 1 * 250 + 80], 
    'Target': 1 # 부상군 정답지
})

# 4. 피지오넷(정상군) 데이터 합치기 (df_31이 메모리에 있다고 가정)
# 만약 df_31이 없다면, 일단 부상자 데이터 내부의 경향성만 먼저 보겠습니다.

# 5. 모델 학습 시작!
X = df_ml_data[['Age', 'Sex', 'VM_at_80']]
y = df_ml_data['Target']

# (참고: 지금은 정상군 데이터가 완벽히 합쳐지지 않아 학습이 제한적일 수 있어요. 
# 일단 데이터가 제대로 불러와지는지부터 확인해볼까요?)

print("✅ 데이터 로드 성공!")
print(df_ml_data.head())

In [ ]:
import scipy.io
import pandas as pd
import numpy as np

# 1. 현경님이 찾아주신 '진짜 경로' 설정
# 경로 앞에 r을 붙여야 역슬래시(\) 에러가 안 납니다!
m_path = r"C:\Downloads\knee_data\Data for Surface EMG muscle activation patterns of the lower extremities during gait in individuals with and without a knee injury history\M.mat"
xlsx_path = r"C:\Downloads\knee_data\Data for Surface EMG muscle activation patterns of the lower extremities during gait in individuals with and without a knee injury history\Anonymized_Data_Identifiers.xlsx"

try:
    # 2. .mat 파일 로드 (M_array 추출)
    mat_contents = scipy.io.loadmat(m_path)
    M_array = mat_contents['M'] # 파일 내 변수명이 'M'인지 확인
    
    # 3. 엑셀 파일 로드 (환자 정보)
    df_info = pd.read_excel(xlsx_path)
    
    # 4. 분석용 마스터 테이블 만들기 (122명 부상자 기준)
    df_final = df_info.iloc[:122].copy()
    
    # 우리가 찾은 '황금 지점' 80pt 데이터 심기
    # M_array 구조: [사람, 근육*250지점] -> 1번 인덱스가 VM(내측광근)
    target_point = 80
    df_final['VM_at_80'] = M_array[:122, 1 * 250 + target_point]
    
    print("✅ 성공! 데이터가 완벽하게 로드되었습니다.")
    print(f"전체 환자 수: {len(df_final)}명")
    print(df_final[['Age at testing [years]', 'VM_at_80']].head())

except FileNotFoundError:
    print("❌ 경로가 아직 조금 이상한 것 같아요. 폴더 이름을 다시 확인해주세요!")
except Exception as e:
    print(f"❌ 에러 발생: {e}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. 학습용 데이터셋 만들기
# 부상자 그룹 (Target: 1)
df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
df_target_1['Target'] = 1

# 정상군 그룹 (피지오넷에서 대응값 추출 - Target: 0)
# 아까 찾은 df_31(또는 df_p)의 LAT.V를 Muscle_Val로 사용합니다.
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_31), # 피지오넷 평균 연령
    'Sex': [0] * len(df_31),
    'Muscle_Val': df_31['LAT.V'],
    'Target': 0
})

# 2. 데이터 합체 (Total 153명)
df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)

# 3. 모델 학습
X = df_ml[['Age', 'Sex', 'Muscle_Val']]
y = df_ml['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. 결과 확인
y_pred = rf_model.predict(X_test)
print(f"✅ 모델 예측 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("-" * 30)
print(classification_report(y_test, y_pred))

# 5. 핵심 기여도 확인 (나이 vs 근육값 중 뭐가 더 중요했나?)
importances = rf_model.feature_importances_
for name, val in zip(X.columns, importances):
    print(f"🔍 {name}의 중요도: {val:.4f}")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. 메모리에서 피지오넷 데이터(TIB.A 컬럼이 있는 DF) 찾기
found_p_name = None
found_p_df = None

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and 'TIB.A' in obj.columns:
        found_p_name = name
        found_p_df = obj.copy()
        break

if found_p_df is not None:
    print(f"✅ 피지오넷 데이터를 찾았습니다! 변수명: '{found_p_name}'")
    
    # 2. 학습용 데이터셋 구성
    # 부상자 그룹 (Target: 1)
    df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_target_1['Target'] = 1

    # 정상군 그룹 (Target: 0) - 찾은 피지오넷 데이터 활용
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(found_p_df), 
        'Sex': [0] * len(found_p_df),
        'Muscle_Val': found_p_df['LAT.V'], # 피지오넷의 외측광근 값을 비교군으로 사용
        'Target': 0
    })

    # 3. 데이터 합체 및 학습
    df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)
    X = df_ml[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml['Target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    # 4. 결과 출력
    y_pred = rf_model.predict(X_test)
    print(f"\n🚀 모델 예측 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
    print("-" * 30)
    print(classification_report(y_test, y_pred))
    
    # 중요도 확인
    for name, val in zip(X.columns, rf_model.feature_importances_):
        print(f"🔍 {name}의 기여도: {val:.4f}")

else:
    print("❌ 피지오넷 데이터를 찾지 못했습니다. 피지오넷 csv를 로드한 셀을 다시 실행해 주세요!")

In [ ]:
# import pandas as pd
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, accuracy_score

# # 1. 피지오넷(정상군) CSV 경로 설정 (현경님 PC의 실제 경로로 수정해주세요!)
# # 예: r"C:\Downloads\physionet_data.csv"
# physio_path = r"피지오넷_CSV_파일의_전체_경로를_여기에_붙여넣으세요"

# try:
#     # 2. 피지오넷 데이터 로드
#     df_p = pd.read_csv(physio_path)
#     print(f"✅ 피지오넷 데이터 로드 성공! (행 수: {len(df_p)})")

#     # 3. 학습용 데이터셋 구성
#     # 부상자 그룹 (Target: 1) - 이전 셀에서 만든 df_final 사용
#     df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
#     df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
#     df_target_1['Target'] = 1

#     # 정상군 그룹 (Target: 0)
#     df_target_0 = pd.DataFrame({
#         'Age': [24] * len(df_p), 
#         'Sex': [0] * len(df_p),
#         'Muscle_Val': df_p['LAT.V'], # 피지오넷의 외측광근 값
#         'Target': 0
#     })

#     # 4. 데이터 합체 및 학습
#     df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)
#     X = df_ml[['Age', 'Sex', 'Muscle_Val']]
#     y = df_ml['Target']

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#     rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
#     rf_model.fit(X_train, y_train)

#     # 5. 결과 출력
#     y_pred = rf_model.predict(X_test)
#     print(f"\n🚀 모델 예측 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
#     print("-" * 30)
#     print(classification_report(y_test, y_pred))
    
#     # 중요도 확인
#     for name, val in zip(X.columns, rf_model.feature_importances_):
#         print(f"🔍 {name}의 기여도: {val:.4f}")

# except Exception as e:
#     print(f"❌ 에러 발생: {e}")
#     print("팁: 피지오넷 CSV 파일 경로가 정확한지, 그리고 이전 셀의 df_final이 잘 만들어졌는지 확인해주세요!")

In [ ]:
df_p

In [5]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. 현경님이 알려주신 피지오넷 데이터 경로
physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

try:
    # 피지오넷 폴더 내의 CSV 파일 중 하나를 샘플로 읽어옵니다 (보통 복수 파일일 수 있음)
    # 여기서는 현경님이 분석하셨던 df_31의 원천 파일을 찾는 로직입니다.
    csv_files = [f for f in os.listdir(physio_folder) if f.endswith('.csv')]
    
    if len(csv_files) > 0:
        # 첫 번째 파일을 대조군으로 사용 (필요시 파일명 특정 가능)
        df_p = pd.read_csv(os.path.join(physio_folder, csv_files[0]))
        print(f"✅ 피지오넷 데이터 로드 성공! 파일명: {csv_files[0]}")

        # 2. 학습 데이터 구성 (df_final은 이전 셀에서 이미 로드됨)
        # 부상자 그룹 (Target: 1)
        df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
        df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
        df_target_1['Target'] = 1

        # 정상군 그룹 (Target: 0)
        # 피지오넷 데이터의 LAT.V를 비교군으로 사용
        df_target_0 = pd.DataFrame({
            'Age': [24] * len(df_p), 
            'Sex': [0] * len(df_p),
            'Muscle_Val': df_p['LAT.V'], 
            'Target': 0
        })

        # 3. 데이터 합체 및 머신러닝
        df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)
        X = df_ml[['Age', 'Sex', 'Muscle_Val']]
        y = df_ml['Target']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_model.fit(X_train, y_train)

        # 4. 최종 결과 출력
        y_pred = rf_model.predict(X_test)
        print(f"\n🎯 무릎 부상 예측 모델 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
        print("-" * 40)
        print(classification_report(y_test, y_pred))
        
        # 기여도 확인
        for name, val in zip(X.columns, rf_model.feature_importances_):
            print(f"🔍 {name}의 예측 기여도: {val:.4f}")
    else:
        print("❌ 폴더 내에 CSV 파일이 보이지 않아요! 확장자를 확인해 주세요.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

✅ 피지오넷 데이터 로드 성공! 파일명: Subject_info.csv
❌ 에러 발생: 'LAT.V'


In [6]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

try:
    # 1. 파일 목록 중 'Subject_info'가 아닌, 실제 데이터 파일을 찾습니다.
    # 보통 파일명에 'walk'가 들어가거나 용량이 큰 파일입니다.
    all_files = os.listdir(physio_folder)
    data_files = [f for f in all_files if 'walk' in f.lower() and f.endswith('.csv')]
    
    if not data_files: # 혹시 walk라는 이름이 없으면 첫 번째 csv라도 시도
        data_files = [f for f in all_files if f.endswith('.csv') and 'info' not in f.lower()]

    # 2. 첫 번째 데이터 파일 로드
    target_file = data_files[0]
    df_p = pd.read_csv(os.path.join(physio_folder, target_file))
    print(f"✅ 데이터 로드 성공! 사용된 파일: {target_file}")
    
    # 3. 컬럼 이름 확인 (LAT.V가 있는지, 아니면 다른 이름인지 체크)
    # 피지오넷 데이터에 따라 'Vastus Lateralis' 등으로 되어 있을 수 있어요.
    possible_cols = ['LAT.V', 'VL', 'Vastus_Lat']
    col_name = next((c for c in possible_cols if c in df_p.columns), None)

    if col_name:
        # --- 여기서부터는 아까와 동일한 머신러닝 코드 ---
        df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
        df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
        df_target_1['Target'] = 1

        df_target_0 = pd.DataFrame({
            'Age': [24] * len(df_p), 
            'Sex': [0] * len(df_p),
            'Muscle_Val': df_p[col_name], 
            'Target': 0
        }).iloc[:122] # 부상자 수와 비슷하게 맞춤

        df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)
        X = df_ml[['Age', 'Sex', 'Muscle_Val']]
        y = df_ml['Target']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_model.fit(X_train, y_train)

        print(f"\n🎯 모델 정확도: {accuracy_score(y_test, rf_model.predict(X_test))*100:.2f}%")
        print(f"🔍 사용된 근육 컬럼: {col_name}")
    else:
        print(f"❌ '{target_file}' 파일 안에 근육 데이터 컬럼이 없어요. 컬럼 목록: {list(df_p.columns[:5])}")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

❌ 에러 발생: list index out of range


In [ ]:
import os

physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

# 폴더 내 모든 파일의 확장자만 추출
extensions = set([os.path.splitext(f)[1] for root, dirs, files in os.walk(physio_folder) for f in files])

print(f"🧐 폴더 내 파일 확장자 목록: {extensions}")

# 파일 샘플 5개만 보기
all_files = [f for root, dirs, files in os.walk(physio_folder) for f in files]
print(f"📋 파일 샘플: {all_files[:5]}")

In [ ]:
import wfdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. 피지오넷 데이터 로드 (S01 파일 기준)
physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"
sample_name = "S01" 
record_path = os.path.join(physio_folder, sample_name)

try:
    # 데이터 읽기
    record = wfdb.rdrecord(record_path)
    df_p = pd.DataFrame(record.p_signal, columns=record.sig_name)
    
    # 2. 분석에 쓸 근육 컬럼 선택 (피지오넷은 보통 'LAT.V'가 외측광근)
    p_col = 'LAT.V' if 'LAT.V' in df_p.columns else df_p.columns[0]
    print(f"✅ 피지오넷 연결 성공! 사용 컬럼: {p_col}")

    # 3. 데이터 통합 (부상자 122명 vs 정상군 122명 샘플링)
    # [부상자 그룹] - 이전 셀의 df_final 활용
    df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_target_1['Target'] = 1

    # [정상군 그룹]
    df_target_0 = pd.DataFrame({
        'Age': [24] * 122, # 대조군 평균 연령 설정
        'Sex': [0] * 122,  # 대조군 성별 설정
        'Muscle_Val': df_p[p_col].iloc[:122].values, # 정상인의 근육 신호
        'Target': 0
    })

    # 최종 머신러닝 데이터셋
    df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)

    # 4. 모델 학습 (Random Forest)
    X = df_ml[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml['Target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # 5. 결과 발표
    y_pred = model.predict(X_test)
    print("\n" + "="*40)
    print(f"🚀 무릎 부상 판독 AI 정확도: {accuracy_score(y_test, y_pred)*100:.2f}%")
    print("="*40)
    print(classification_report(y_test, y_pred))

    # 어떤 요소가 가장 중요했나?
    importances = pd.Series(model.feature_importances_, index=X.columns)
    print("\n📊 분석 기여도(Feature Importance):")
    print(importances.sort_values(ascending=False))

except Exception as e:
    print(f"❌ 최종 실행 에러: {e}")
    print("팁: 혹시 'S01' 파일이 폴더 바로 아래에 있는지 확인해 주세요!")
    # 정규화 학인 코드

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. 스케일러 준비
scaler = StandardScaler()

# 2. 피지오넷에서 실제 근육 컬럼 다시 찾기 (baso 대신 EMG 관련 확인)
# record.sig_name 목록 중 'LAT.V', 'VL', 'RF' 등이 있는지 확인이 필요해요!
target_muscle = 'LAT.V' if 'LAT.V' in df_p.columns else df_p.columns[0] 

# 3. 데이터 다시 구성
df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
df_target_1['Target'] = 1

df_target_0 = pd.DataFrame({
    'Age': [24] * 122,
    'Sex': [0] * 122,
    'Muscle_Val': df_p[target_muscle].iloc[:122].values,
    'Target': 0
})

df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)

# 4. 정규화 적용 (Age와 Muscle_Val의 단위를 통일)
df_ml[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml[['Age', 'Muscle_Val']])

# 5. 모델 재학습
X = df_ml[['Age', 'Sex', 'Muscle_Val']]
y = df_ml['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print(f"📊 정규화 후 정확도: {rf_model.score(X_test, y_test)*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# 부상자 그룹 분포
sns.kdeplot(df_ml[df_ml['Target'] == 1]['Muscle_Val'], label='Injured (VM_at_80)', shade=True)

# 정상군 그룹 분포
sns.kdeplot(df_ml[df_ml['Target'] == 0]['Muscle_Val'], label='Normal (PhysioNet)', shade=True)

plt.title('Data Distribution: Injured vs Normal')
plt.xlabel('Normalized Muscle Value')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# 피지오넷 파일에 들어있는 진짜 컬럼(신호) 이름들 확인
print("📋 피지오넷 사용 가능 컬럼 목록:")
for i, name in enumerate(record.sig_name):
    print(f"{i}: {name}")

In [ ]:
# 1. 컬럼명을 진짜 근육 신호로 변경
# 현경님의 데이터(VM)와 가장 유사한 대조군인 '외측광근(LAT.V)'을 선택합니다.
target_muscle = 'semg LT LAT.V' 

# 2. 데이터 다시 구성
df_target_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_target_1.columns = ['Age', 'Sex', 'Muscle_Val']
df_target_1['Target'] = 1

df_target_0 = pd.DataFrame({
    'Age': [24] * 122,
    'Sex': [0] * 122,
    'Muscle_Val': df_p[target_muscle].iloc[:122].values, # 진짜 근육 신호 투입!
    'Target': 0
})

df_ml = pd.concat([df_target_1, df_target_0], axis=0).reset_index(drop=True)

# 3. 정규화 및 모델 학습
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_ml[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml[['Age', 'Muscle_Val']])

X = df_ml[['Age', 'Sex', 'Muscle_Val']]
y = df_ml['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. 분포 그래프 다시 그리기 (진실 확인용)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.kdeplot(df_ml[df_ml['Target'] == 1]['Muscle_Val'], label='Injured (VM)', shade=True)
sns.kdeplot(df_ml[df_ml['Target'] == 0]['Muscle_Val'], label='Normal (LAT.V)', shade=True)
plt.title('Corrected Distribution: Muscle vs Muscle')
plt.legend()
plt.show()

print(f"📊 근육 대조 후 정확도: {rf_model.score(X_test, y_test)*100:.2f}%")

In [ ]:
import numpy as np

# 1. RMS(Root Mean Square) 변환 함수
def to_rms(signal, window_size=50):
    return np.sqrt(np.convolve(signal**2, np.ones(window_size)/window_size, mode='same'))

# 2. 피지오넷 신호를 RMS로 변환하여 부상자 데이터와 수준 맞추기
target_muscle = 'semg LT LAT.V'
physio_rms = to_rms(df_p[target_muscle].values)

# 3. 데이터 다시 구성
df_target_0 = pd.DataFrame({
    'Age': [24] * 122,
    'Sex': [0] * 122,
    'Muscle_Val': physio_rms[:122], # 변환된 부드러운 신호 투입!
    'Target': 0
})

# (이후 모델 학습 및 그래프 코드는 동일하게 실행)

In [ ]:
import wfdb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. 경로 설정
physio_folder = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

# 2. 정상군 데이터 수집 (속도를 위해 S01~S05 5명만 먼저 수집해 봅시다!)
normal_vals = []
for i in range(1, 6):
    pid = f"S{str(i).zfill(2)}"
    try:
        record = wfdb.rdrecord(os.path.join(physio_folder, pid))
        sig = record.p_signal[:, record.sig_name.index('semg LT LAT.V')]
        # RMS 변환
        rms = np.sqrt(np.convolve(sig**2, np.ones(50)/50, mode='same'))
        # 데이터 25개씩 샘플링
        normal_vals.extend(np.random.choice(rms, 25))
    except: continue

# 3. 데이터셋 합치기
# [부상자] - 현경님이 만드신 데이터
df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
df_t1['Target'] = 1

# [정상군] - 방금 뽑은 피지오넷 데이터
df_t0 = pd.DataFrame({
    'Age': [24] * len(normal_vals),
    'Sex': [0] * len(normal_vals),
    'Muscle_Val': normal_vals,
    'Target': 0
}).iloc[:len(df_t1)] # 개수 맞추기

df_ml = pd.concat([df_t1, df_t0]).reset_index(drop=True)

# 4. 정규화 및 모델링
scaler = StandardScaler()
df_ml[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml[['Age', 'Muscle_Val']])

X = df_ml[['Age', 'Sex', 'Muscle_Val']]
y = df_ml['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 5. 시각화 (이게 떠야 성공!)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_ml, x='Muscle_Val', hue='Target', fill=True)
plt.title('Final Distribution Check: Injured(1) vs Normal(0)')
plt.show()

print(f"🎯 최종 모델 정확도: {rf.score(X_test, y_test)*100:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. df_cleaned에서 정상군 데이터 추출
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_cleaned),      # 정상군 평균 연령 (임의 설정)
    'Sex': [0] * len(df_cleaned),       # 정상군 성별 (임의 설정)
    'Muscle_Val': df_cleaned['LAT.V'],  # 정제된 외측광근 평균값
    'Target': 0                         # 정상군 라벨
})

# 2. 부상자 데이터와 합치기 (df_t1은 아까 만든 부상자 데이터)
df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
df_t1['Target'] = 1

df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

# 3. 정규화 (스케일링)
scaler = StandardScaler()
df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

# 4. 모델 학습
X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
y = df_ml_final['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

# 5. 분포 확인 그래프 (이번엔 진짜 겹쳐야 합니다!)
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (VM)', fill=True, alpha=0.5)
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (LAT.V)', fill=True, alpha=0.5)
plt.title("Final Comparison: Injured vs Cleaned Normal Data")
plt.legend()
plt.show()

print(f"🎯 최종 모델 정확도: {rf_final.score(X_test, y_test)*100:.2f}%")

In [ ]:
# --- [정상군 데이터 구성] ---
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_cleaned),      # 정상군 연령
    'Sex': [0] * len(df_cleaned),       # 정상군 성별
    'Muscle_Val': df_cleaned['LAT.V'],  # 정제된 외측광근 평균값
    'Target': 0
})

# --- [부상자 데이터 구성] ---
# df_final이 메모리에 있는지 확인하며 합칩니다.
df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
df_t1['Target'] = 1

# --- [데이터 통합 및 스케일링] ---
df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

# --- [최종 모델 학습] ---
X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
y = df_ml_final['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

# --- [결과 확인] ---
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
plt.title("Final Match: Injured vs Cleaned Normal Data")
plt.show()

print(f"🎯 드디어 연결 성공! 최종 정확도: {rf_final.score(X_test, y_test)*100:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 정상군 데이터 구성 (PhysioNet의 LAT.V 활용)
# df_cleaned가 위에서 확인된 상태이므로 바로 사용합니다.
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_cleaned),      # 정상군 평균 연령 (임의 설정)
    'Sex': [0] * len(df_cleaned),       # 정상군 성별 (임의 설정)
    'Muscle_Val': df_cleaned['LAT.V'],  # 이미 확인된 LAT.V 값 사용
    'Target': 0                         # 정상군 라벨
})

# 2. 부상자 데이터 구성 (기존 df_final 활용)
# 혹시 df_final 이름이 바뀌었다면 확인이 필요합니다.
df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
df_t1['Target'] = 1

# 3. 데이터 통합
df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

# 4. 정규화 (데이터 단위 맞추기)
scaler = StandardScaler()
df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

# 5. 모델 학습 및 평가
X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
y = df_ml_final['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_train, y_train)

# 6. 결과 시각화 (두 산이 만나는지 확인!)
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (VM)', fill=True, alpha=0.5)
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (LAT.V)', fill=True, alpha=0.5)
plt.title("Final Comparison: Muscle Activation Distribution")
plt.legend()
plt.show()

print(f"🎯 모델 분석 완료! 최종 정확도: {rf_final.score(X_test, y_test)*100:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# [핵심] 현재 메모리에 있는 df_cleaned를 강제로 다시 인식시킵니다.
try:
    # 1. 데이터 확인 (정상군)
    print("✅ 데이터 확인 중...")
    temp_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 2. 부상자 데이터 (df_final이 살아있는지 확인)
    temp_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    temp_1.columns = ['Age', 'Sex', 'Muscle_Val']
    temp_1['Target'] = 1

    # 3. 데이터 통합
    df_ml_final = pd.concat([temp_1, temp_0]).reset_index(drop=True)

    # 4. 정규화 및 학습
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
    
    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 5. 그래프 그리기
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
    plt.title("Finally! Muscle Data Distribution")
    plt.show()

    print(f"🎯 성공! 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")

except NameError as e:
    print(f"❌ 아직도 이름을 못 찾네요: {e}")
    print("💡 해결법: 위쪽 'df_cleaned'가 출력되었던 셀을 다시 한번 [Shift+Enter] 누르고 얘를 돌려보세요!")

In [ ]:
# 1. 아까 그 31명 데이터(df_cleaned)를 여기서 바로 다시 정의하거나 
# 혹은 아까 df_cleaned를 만들었던 코드를 이 위에 살짝 얹어주세요.

# (만약 위에서 df_cleaned가 정의된 셀이 있다면, 그 셀 번호를 확인해보세요. 
# 그 셀을 실행한 직후에 바로 아래 코드를 실행해야 합니다!)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# [중요] 만약 df_cleaned를 못 찾으면, 강제로 만들어버리기
# ---------------------------------------------------------
if 'df_cleaned' not in locals():
    print("⚠️ df_cleaned를 메모리에서 찾을 수 없습니다.")
    print("💡 해결책: 31명 그래프 그렸던 셀을 다시 [Shift+Enter] 누르고 오세요!")
else:
    # 1. 정상군 데이터 (PhysioNet)
    temp_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 2. 부상자 데이터 (df_final)
    temp_1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    temp_1.columns = ['Age', 'Sex', 'Muscle_Val']
    temp_1['Target'] = 1

    # 3. 데이터 통합
    df_ml_final = pd.concat([temp_1, temp_0]).reset_index(drop=True)

    # 4. 정규화 및 학습
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
    
    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 5. 그래프
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
    plt.title("Finally! Success!")
    plt.show()

    print(f"🎯 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. [중요] df_cleaned를 모델이 쓸 수 있게 다시 한번 확정해줍니다.
# (현경님이 방금 보여주신 그 데이터를 기반으로 합니다)
try:
    # 정상군 데이터셋 만들기 (df_cleaned의 LAT.V 컬럼 사용)
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 2. 부상자 데이터셋 만들기 (df_final 사용)
    # 혹시 df_final 컬럼명이 다르면 여기서 에러가 날 수 있으니 이름을 맞춰줍니다.
    df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_t1['Target'] = 1

    # 3. 두 데이터 합치기
    df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

    # 4. 머신러닝을 위한 정규화
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

    # 5. 모델 학습 시작!
    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_final.fit(X_train, y_train)

    # 6. 결과 시각화 (두 산이 만나는지 확인!)
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True, common_norm=False)
    plt.title("Success! Injured(1) vs Normal(0) Distribution")
    plt.show()

    print(f"🎯 드디어 성공! 모델 정확도: {rf_final.score(X_test, y_test)*100:.2f}%")

except Exception as e:
    print(f"❌ 아이구, 또 에러가 났네요: {e}")
    print("💡 팁: 이 셀을 돌리기 바로 직전에 'df_cleaned'가 출력된 [26]번 셀을 한번 더 실행해주세요!")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 메모리에서 피지오넷 데이터 찾기
df_cleaned = None  # 우리가 찾던 그 이름으로 미리 준비!

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if 'TIB.A' in obj.columns:
            df_cleaned = obj.copy() # 찾은 데이터를 df_cleaned라는 이름으로 '복사'해서 저장!
            print(f"✅ 드디어 찾았습니다! 이제부터 이 데이터의 이름은 'df_cleaned' 입니다.")
            break

if df_cleaned is not None:
    # 2. 비율 계산 및 히트맵 (현경님 기존 로직)
    df_p = df_cleaned.copy()
    df_p['REC_HAM_Ratio'] = df_p['REC.F'] / (df_p['HAM'] + 1e-9)
    
    cols = ['TIB.A', 'LAT.G', 'REC.F', 'HAM', 'LAT.V', 'REC_HAM_Ratio']
    plt.figure(figsize=(10, 8))
    sns.heatmap(df_p[cols].corr(), annot=True, fmt=".2f", cmap='RdBu_r', center=0)
    plt.title("PhysioNet Data Correlation (Fixed Name)", fontsize=15)
    plt.show()
    
    print("🚀 이제 바로 아래 셀에서 아까 그 'RandomForest 모델 코드'를 다시 돌려보세요!")
else:
    print("❌ 'TIB.A' 컬럼을 가진 표를 여전히 찾지 못했습니다. 원본 데이터를 로드하는 셀을 먼저 실행해 주세요!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 메모리를 뒤져서 아까 로드한 31명 데이터를 'df_cleaned'로 강제 지정
found_df = None
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and 'TIB.A' in obj.columns:
        found_df = obj
        break

if found_df is not None:
    df_cleaned = found_df.copy()
    print("✅ 피지오넷 데이터(df_cleaned)를 성공적으로 연결했습니다!")
    
    # 2. 정상군 데이터셋 (Target 0)
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 3. 부상자 데이터셋 (Target 1 - df_final 기반)
    # (df_final이 없는 경우를 대비해 try-except 사용)
    try:
        df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
        df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
        df_t1['Target'] = 1
        
        # 4. 데이터 통합
        df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)
        
        # 5. 정규화 및 모델 학습
        scaler = StandardScaler()
        df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
        
        X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
        y = df_ml_final['Target']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train, y_train)
        
        # 6. 대망의 그래프!
        plt.figure(figsize=(10, 6))
        sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True, common_norm=False)
        plt.title("Success! Injured vs Normal Muscle Activation")
        plt.show()
        
        print(f"🎯 드디어! 모델 정확도: {rf.score(X_test, y_test)*100:.2f}%")
        
    except NameError:
        print("❌ 'df_final'(부상자 데이터)을 찾을 수 없습니다. 부상자 엑셀을 불러온 셀을 실행해 주세요!")
else:
    print("❌ 피지오넷 데이터를 아직 못 찾았습니다. [3], [4]번 셀을 다시 실행하고 와주세요!")

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 데이터를 하나로 합치기 (31명 데이터 수집)
all_data = []
target_muscles = {'TIB.A': 2, 'LAT.G': 3, 'REC.F': 4, 'HAM': 5, 'LAT.V': 6}

print("⏳ 31명의 데이터를 수집 중입니다... 잠시만 기다려 주세요.")

for file in file_list: # [3]번 셀에서 만든 file_list 사용
    try:
        record = wfdb.rdrecord(file.replace('.dat', ''))
        df_temp = pd.DataFrame(record.p_signal, columns=record.sig_name)
        
        # 근육별 평균값(Mean Amplitude) 계산
        muscle_means = {m: np.mean(np.abs(df_temp[m])) for m in target_muscles.keys() if m in df_temp.columns}
        all_data.append(muscle_means)
    except:
        continue

df_cleaned = pd.DataFrame(all_data) # 드디어 df_cleaned 탄생!

if not df_cleaned.empty:
    print(f"✅ {len(df_cleaned)}명의 데이터 통합 완료!")
    
    # 2. 정상군(Target 0) vs 부상자(Target 1) 데이터 합치기
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 부상자 데이터 (df_final) 연결
    df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_t1['Target'] = 1

    df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

    # 3. 모델 학습
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 4. 그래프 출력
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
    plt.title("Finally Finished! Muscle Activation Distribution")
    plt.show()

    print(f"🎯 모델 정확도: {rf.score(X_test, y_test)*100:.2f}%")
else:
    print("❌ 데이터를 합치지 못했습니다. base_path 경로를 다시 확인해 주세요!")

In [ ]:
import wfdb
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 경로 설정 (현경님 컴퓨터 경로 그대로)
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-sub"
file_list = sorted(glob.glob(os.path.join(base_path, "*.dat")))

print(f"✅ 찾은 피험자 수: {len(file_list)}명")

# 2. 데이터 수집 및 df_cleaned 만들기
all_results = []
target_muscles = {'TIB.A': 2, 'LAT.G': 3, 'REC.F': 4, 'HAM': 5, 'LAT.V': 6}

print("⏳ 데이터를 분석 중입니다... (약 10~20초 소요)")
for file in file_list:
    try:
        record = wfdb.rdrecord(file.replace('.dat', ''))
        df_temp = pd.DataFrame(record.p_signal, columns=record.sig_name)
        # 근육별 평균 절대값 계산
        muscle_means = {m: np.mean(np.abs(df_temp[m])) for m in target_muscles.keys() if m in df_temp.columns}
        all_results.append(muscle_means)
    except:
        continue

df_cleaned = pd.DataFrame(all_results)

# 3. 모델 학습 및 그래프 출력
if not df_cleaned.empty:
    # 정상군(Target 0)
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned['LAT.V'],
        'Target': 0
    })

    # 부상자(Target 1) - df_final이 메모리에 있어야 합니다!
    try:
        df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
        df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
        df_t1['Target'] = 1
        
        df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)
        
        # 학습용 정규화
        scaler = StandardScaler()
        df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
        
        X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
        y = df_ml_final['Target']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train, y_train)
        
        # 결과 시각화
        plt.figure(figsize=(10, 6))
        sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
        plt.title("Injured vs Normal: Final Model Result")
        plt.show()
        
        print(f"🎯 드디어 성공! 모델 정확도: {rf.score(X_test, y_test)*100:.2f}%")
        
    except NameError:
        print("❌ 'df_final'(부상자 엑셀 데이터)을 못 찾겠어요! 엑셀 불러오는 셀만 한번 실행하고 다시 오세요!")
else:
    print("❌ 데이터를 수집하지 못했습니다. base_path 경로에 파일이 있는지 확인해주세요!")

In [ ]:
import glob
import os

# 1. 경로 설정 (주소창 복사한 경로 그대로!)
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-sub"

# 2. 하위 폴더(recursive=True)까지 싹 다 뒤져서 .dat 파일 찾기
# 경로 끝에 /**/*.dat 를 붙이면 모든 하위 폴더를 다 검사합니다.
file_list = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)

print(f"🔎 확인된 총 파일 개수: {len(file_list)}개")

if len(file_list) > 0:
    print("✅ 드디어 파일을 찾았습니다! 이제 통합 모델 코드를 돌리시면 됩니다.")
    # 샘플로 첫 번째 파일 경로 출력해서 확인
    print(f"샘플 경로: {file_list[0]}")
else:
    print("❌ 여전히 0개입니다. 혹시 폴더 경로를 'surface-...' 폴더까지만 잡으셨나요?")
    print("폴더를 열자마자 S01, S02 폴더들이 보이는지 확인해 주세요!")

In [ ]:
import wfdb
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 경로 설정 (버전 정보 포함된 정확한 경로)
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"

# 2. 하위 폴더까지 싹 다 뒤져서 .dat 파일 찾기
file_list = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)
print(f"✅ 찾은 총 파일 개수: {len(file_list)}개")

# 3. 데이터 로드 및 정상군(df_cleaned) 만들기
all_results = []
target_muscles = {'TIB.A': 2, 'LAT.G': 3, 'REC.F': 4, 'HAM': 5, 'LAT.V': 6}

if len(file_list) > 0:
    print("⏳ 데이터 분석 중... 잠시만 기다려 주세요.")
    for file in file_list[:31]: # 31명 샘플링
        try:
            record = wfdb.rdrecord(file.replace('.dat', ''))
            df_temp = pd.DataFrame(record.p_signal, columns=record.sig_name)
            # 평균 절대값 계산
            m_means = {m: np.mean(np.abs(df_temp[m])) for m in target_muscles.keys() if m in df_temp.columns}
            all_results.append(m_means)
        except:
            continue
    
    df_cleaned = pd.DataFrame(all_results)

    # 4. 모델 학습 (정상 vs 부상)
    if not df_cleaned.empty:
        df_target_0 = pd.DataFrame({
            'Age': [24] * len(df_cleaned),
            'Sex': [0] * len(df_cleaned),
            'Muscle_Val': df_cleaned['LAT.V'],
            'Target': 0
        })

        try:
            # 부상자 데이터 (df_final이 메모리에 있어야 함)
            df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
            df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
            df_t1['Target'] = 1
            
            df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)
            
            scaler = StandardScaler()
            df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])
            
            X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
            y = df_ml_final['Target']
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
            
            rf = RandomForestClassifier(n_estimators=100, random_state=42)
            rf.fit(X_train, y_train)
            
            # 시각화
            plt.figure(figsize=(10, 6))
            sns.kdeplot(data=df_ml_final, x='Muscle_Val', hue='Target', fill=True)
            plt.title("Injured vs Normal: Success!")
            plt.show()
            
            print(f"🎯 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")
            
        except NameError:
            print("❌ 부상자 데이터(df_final)를 못 찾았어요! 엑셀 불러온 셀을 한 번만 실행하고 다시 오세요.")
else:
    print("❌ 여전히 파일을 못 찾았어요. 폴더 경로를 다시 확인해 주세요!")

In [ ]:
# 현재 데이터가 잘 만들어졌는지 확인하는 코드
if 'df_cleaned' in globals():
    print(f"✅ 데이터가 잘 살아있습니다! 총 {len(df_cleaned)}명의 데이터가 수집되었네요.")
    print("\n📊 수집된 근육 데이터 샘플:")
    display(df_cleaned.head())
else:
    print("❌ 데이터가 비어있습니다. 아까 그 긴 통합 코드를 '끝까지' 다시 복사해서 돌려주세요!")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 정상군(피지오넷) 데이터 정리
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_cleaned),
    'Sex': [0] * len(df_cleaned),
    'Muscle_Val': df_cleaned['LAT.V'],
    'Target': 0
})

# 2. 부상자(엑셀) 데이터 정리
df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
df_t1['Target'] = 1

# 3. 통합 데이터셋 만들기
df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

# 4. 모델 학습을 위한 전처리
scaler = StandardScaler()
df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
y = df_ml_final['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. RandomForest 모델 실행
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 6. 대망의 결과 시각화
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (PhysioNet)', fill=True, color='blue')
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (Excel)', fill=True, color='orange')
plt.title("Success! Final Muscle Activation Analysis", fontsize=15)
plt.legend()
plt.show()

print(f"🎯 현경님이 만든 모델의 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. LAT.V 근육의 정확한 컬럼명 찾기 (대소문자/오타 방지)
actual_col = None
for col in df_cleaned.columns:
    if 'LAT.V' in col.upper() or 'LATV' in col.upper():
        actual_col = col
        break

if actual_col:
    print(f"✅ 근육 데이터를 찾았습니다! 사용된 컬럼명: '{actual_col}'")
    
    # 2. 정상군(피지오넷) 데이터 정리
    df_target_0 = pd.DataFrame({
        'Age': [24] * len(df_cleaned),
        'Sex': [0] * len(df_cleaned),
        'Muscle_Val': df_cleaned[actual_col],
        'Target': 0
    })

    # 3. 부상자(엑셀) 데이터 정리
    df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_t1['Target'] = 1

    # 4. 통합 데이터셋 만들기
    df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

    # 5. 전처리 및 모델 학습
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 6. 결과 시각화
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (PhysioNet)', fill=True, color='blue')
    sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (Excel)', fill=True, color='orange')
    plt.title("Success! Final Muscle Activation Analysis", fontsize=15)
    plt.legend()
    plt.show()

    print(f"🎯 현경님의 AI 모델 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")
else:
    print("❌ 근육 데이터를 찾을 수 없습니다. df_cleaned의 컬럼 목록을 확인해봐야 할 것 같아요!")
    print

In [ ]:
# 지금 df_cleaned에 들어있는 진짜 이름이 뭔지 확인하는 코드
print("--- 현재 데이터에 들어있는 근육 이름들 ---")
print(df_cleaned.columns.tolist())
print("---------------------------------------")

# 만약 데이터가 아예 텅 비어있는지도 확인
if df_cleaned.empty:
    print("⚠️ 어라? 데이터 표 자체가 비어있네요. 데이터 수집 단계에서 문제가 있었나봐요!")
else:
    print("✅ 데이터는 무사합니다! 이름만 맞추면 바로 그래프 그릴 수 있어요.")

In [ ]:
import wfdb
import os
import glob
import pandas as pd
import numpy as np

# 1. 경로 재설정
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"
file_list = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)

all_results = []
print(f"⏳ 총 {len(file_list)}개 파일을 하나씩 강제로 엽니다...")

for file in file_list[:31]:
    try:
        # 확장자 떼고 이름만 추출
        record_name = os.path.splitext(file)[0]
        record = wfdb.rdrecord(record_name)
        
        # 데이터가 있으면 첫 번째 근육(보통 이게 메인입니다) 데이터 추출
        if record.p_signal.shape[1] > 0:
            val = np.mean(np.abs(record.p_signal[:, 0])) # 첫 번째 열의 평균 절대값
            all_results.append({'Muscle_Val': val})
    except Exception as e:
        continue

# 데이터프레임 강제 생성
df_cleaned = pd.DataFrame(all_results)

if not df_cleaned.empty:
    print(f"✅ 성공!!! {len(df_cleaned)}명의 데이터를 구출했습니다!")
    print("이제 아까 '최종 피날레 코드'를 다시 돌리시면 그래프가 뜹니다!")
else:
    print("❌ 파일은 있는데 내용물을 못 읽고 있어요. 폴더 안에 .hea 파일도 같이 있는지 확인해주세요!")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 방금 구출한 31명 데이터 (정상군)
df_target_0 = pd.DataFrame({
    'Age': [24] * len(df_cleaned),
    'Sex': [0] * len(df_cleaned),
    'Muscle_Val': df_cleaned['Muscle_Val'], # 구출한 데이터 이름
    'Target': 0
})

# 2. 엑셀에서 가져온 부상자 데이터 (df_final이 살아있어야 함)
try:
    df_t1 = df_final[['Age at testing [years]', 'Sex (0-Female, 1-Male)', 'VM_at_80']].copy()
    df_t1.columns = ['Age', 'Sex', 'Muscle_Val']
    df_t1['Target'] = 1

    # 3. 데이터 합치기
    df_ml_final = pd.concat([df_t1, df_target_0]).reset_index(drop=True)

    # 4. 전처리 및 AI 학습
    scaler = StandardScaler()
    df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

    X = df_ml_final[['Age', 'Sex', 'Muscle_Val']]
    y = df_ml_final['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 5. 결과 그래프 그리기
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (PhysioNet)', fill=True, color='blue')
    sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (Excel)', fill=True, color='orange')
    plt.title("FINAL SUCCESS: Injured vs Normal Muscle Analysis", fontsize=15)
    plt.xlabel("Standardized Muscle Activation")
    plt.legend()
    plt.show()

    print(f"🎯 현경님, AI 모델의 최종 정확도입니다: {rf.score(X_test, y_test)*100:.2f}%")

except NameError:
    print("❌ 앗! 부상자 데이터(df_final)가 든 엑셀 셀을 한 번만 다시 실행하고 와주세요!")

In [ ]:
# 기존 모델 코드 중 전처리 부분만 살짝 수정해서 다시 그려봅시다!
from sklearn.preprocessing import RobustScaler

# 1. 이상치에 강한 스케일러로 교체
scaler = RobustScaler()
df_ml_final[['Age', 'Muscle_Val']] = scaler.fit_transform(df_ml_final[['Age', 'Muscle_Val']])

# 2. 그래프 다시 그리기 (로그 스케일 적용 없이도 잘 보이게!)
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Val', label='Normal (PhysioNet)', fill=True, color='blue', bw_adjust=1.5)
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Val', label='Injured (Excel)', fill=True, color='orange', bw_adjust=1.5)

plt.title("Fixed Scale: Normal vs Injured Comparison", fontsize=15)
plt.xlabel("Normalized Muscle Value (Standardized)")
# x축 범위를 데이터가 있는 곳 위주로 자동 조정
plt.xlim(df_ml_final['Muscle_Val'].min(), df_ml_final['Muscle_Val'].max()) 
plt.legend()
plt.show()

print(f"🎯 스케일 조정 후 최종 정확도: {rf.score(X_test, y_test)*100:.2f}%")

In [ ]:
import numpy as np

# 1. 시각화를 위해 데이터 스케일을 로그 변환 (단위 차이가 너무 클 때 쓰는 마법입니다)
df_ml_final['Muscle_Log'] = np.log1p(df_ml_final['Muscle_Val'])

# 2. 그래프 그리기
plt.figure(figsize=(12, 6))

# 정상군과 부상자 각각 그리기
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==0], x='Muscle_Log', 
            label='Normal (PhysioNet)', fill=True, color='blue', bw_adjust=1)
sns.kdeplot(data=df_ml_final[df_ml_final['Target']==1], x='Muscle_Log', 
            label='Injured (Excel)', fill=True, color='orange', bw_adjust=1)

plt.title("Clinical Analysis: Normal vs Injured (Log Scale)", fontsize=15)
plt.xlabel("Log-transformed Muscle Activation")
plt.ylabel("Density")
plt.legend()
plt.show()

# 어떤 변수가 가장 중요했는지 확인 (피처 중요도)
importances = rf.feature_importances_
feature_names = ['Age', 'Sex', 'Muscle_Val']
for name, importance in zip(feature_names, importances):
    print(f"🔍 {name}의 중요도: {importance:.4f}")

In [ ]:
import wfdb
import os
import glob
import pandas as pd
import numpy as np

# 1. 다시 파일 목록 훑기
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"
file_list = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)

multi_results = []
print(f"⏳ 31개 파일에서 모든 근육 신호를 구출하는 중...")

for file in file_list[:31]:
    try:
        record_name = os.path.splitext(file)[0]
        record = wfdb.rdrecord(record_name)
        
        # 사용 가능한 근육 이름들 확인 (예: 'LAT.V', 'V.MED', 'TIB.A' 등)
        temp_data = {'Age': 24, 'Sex': 0, 'Target': 0}
        for i, sig_name in enumerate(record.sig_name):
            # 각 근육 신호의 평균 절대값을 추출
            temp_data[sig_name] = np.mean(np.abs(record.p_signal[:, i]))
        
        multi_results.append(temp_data)
    except:
        continue

df_multi_normal = pd.DataFrame(multi_results)
print(f"✅ 구출 성공! 추출된 근육 목록: {df_multi_normal.columns.tolist()}")

# 2. 엑셀 데이터(부상자)와 합치기 (이름이 매칭되는 것 위주로)
# 엑셀의 VM_at_80을 V.MED 혹은 LAT.V와 비교하도록 설정 가능합니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 시각화할 근육 리스트만 쏙 뽑기 (semg로 시작하는 것들)
muscle_cols = [col for col in df_multi_normal.columns if 'semg' in col]

# 2. 모든 근육의 평균값 계산
muscle_means = df_multi_normal[muscle_cols].mean().sort_values(ascending=False)

# 3. 그래프 그리기
plt.figure(figsize=(12, 8))
sns.barplot(x=muscle_means.values, y=muscle_means.index, palette='viridis')
plt.title("Average Muscle Activation (Normal Subjects)", fontsize=15)
plt.xlabel("Mean Absolute Value")
plt.ylabel("Muscle Groups")
plt.show()

print("✅ 정상군 31명이 보행 시 어떤 근육을 가장 많이 쓰는지 순위가 나왔습니다!")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 좌우 쌍이 맞는 근육들 추출
pairs = ['TIB.A', 'LAT.V', 'LAT.G', 'REC.F', 'HAM']
asymmetry_data = []

for _, row in df_multi_normal.iterrows():
    diffs = {'Target': 0}
    for muscle in pairs:
        # 좌우 차이의 절대값 계산 (누가 더 붉균형한가?)
        rt_val = row.get(f'semg RT {muscle}', 0)
        lt_val = row.get(f'semg LT {muscle}', 0)
        diffs[f'{muscle}_diff'] = abs(rt_val - lt_val)
    asymmetry_data.append(diffs)

df_asym_normal = pd.DataFrame(asymmetry_data)

# 2. 부상자(엑셀) 데이터와 비교 (엑셀에 좌우 데이터가 있다면 더 좋지만, 없다면 정상군의 불균형 분포만 먼저 확인)
plt.figure(figsize=(12, 6))
df_plot = df_asym_normal.drop('Target', axis=1)
sns.boxplot(data=df_plot, palette='Set3')
plt.title("Asymmetry Index in Normal Group (RT vs LT Difference)", fontsize=15)
plt.ylabel("Difference Value (Absolute)")
plt.show()

print("✅ 정상군 31명의 근육별 좌우 불균형 분포입니다.")

In [ ]:
# 엑셀 데이터에서 환자별 좌우 차이 계산하기
asym_injured = []

# ID별로 그룹화해서 안 아픈 다리(0)와 아픈 다리(1)의 차이 구하기
for subject_id, group in df_info.groupby('Subject ID'):
    if len(group) >= 2:
        # VM_at_80 값 (혹은 다른 근육값)의 차이 계산
        # 현재 df_info에는 근육값이 합쳐져 있어야 하므로, 
        # 이전에 작업한 df_final(근육값 포함)을 기준으로 계산하는게 좋습니다.
        pass

print("✅ 환자별 좌우 불균형을 계산할 준비가 되었습니다!")

In [ ]:
# 1. 부상자 그룹(엑셀)의 좌우 불균형 계산
injured_diffs = []
# df_final(근육 데이터 포함)을 Subject ID로 그룹화
for sub_id, group in df_final.groupby('Subject ID'):
    if len(group) >= 2:
        # 아픈 다리(1)와 안 아픈 다리(0) 수치 가져오기
        val_injured = group[group['Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)'] == 1]['VM_at_80'].values
        val_normal = group[group['Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)'] == 0]['VM_at_80'].values
        
        if len(val_injured) > 0 and len(val_normal) > 0:
            diff = abs(val_injured[0] - val_normal[0])
            injured_diffs.append(diff)

# 2. 데이터 합치기 (시각화용)
normal_diffs = df_asym_normal['LAT.V_diff'].tolist() # 아까 구한 정상군 외측광근 차이

plot_data = pd.DataFrame({
    'Group': ['Normal'] * len(normal_diffs) + ['Injured'] * len(injured_diffs),
    'Asymmetry_Index': normal_diffs + injured_diffs
})

# 3. 운명의 비교 그래프 출력
plt.figure(figsize=(8, 6))
sns.boxplot(data=plot_data, x='Group', y='Asymmetry_Index', palette={'Normal': 'skyblue', 'Injured': 'salmon'})
sns.stripplot(data=plot_data, x='Group', y='Asymmetry_Index', color='black', alpha=0.3) # 개별 데이터 점 찍기

plt.title("Comparison of Muscle Asymmetry: Normal vs Injured", fontsize=15)
plt.ylabel("Left-Right Difference (Activation)")
plt.show()

print(f"📊 분석 결과: 정상군 평균 불균형 대비 부상자 그룹의 불균형 정도를 시각화했습니다.")

In [ ]:
# 1. 비율 기반 불균형 계산 함수
def get_asym_ratio(val1, val2):
    return abs(val1 - val2) / (val1 + val2) * 100

# 2. 정상군 비율 계산
normal_ratios = []
for _, row in df_multi_normal.iterrows():
    rt = row.get('semg RT LAT.V', 0)
    lt = row.get('semg LT LAT.V', 0)
    if (rt + lt) > 0:
        normal_ratios.append(get_asym_ratio(rt, lt))

# 3. 부상자 그룹 비율 계산
injured_ratios = []
for sub_id, group in df_final.groupby('Subject ID'):
    if len(group) >= 2:
        v_inj = group[group['Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)'] == 1]['VM_at_80'].values
        v_con = group[group['Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)'] == 0]['VM_at_80'].values
        if len(v_inj) > 0 and len(v_con) > 0:
            injured_ratios.append(get_asym_ratio(v_inj[0], v_con[0]))

# 4. 시각화
ratio_plot_data = pd.DataFrame({
    'Group': ['Normal'] * len(normal_ratios) + ['Injured'] * len(injured_ratios),
    'Asymmetry_Ratio(%)': normal_ratios + injured_ratios
})

plt.figure(figsize=(10, 6))
sns.violinplot(data=ratio_plot_data, x='Group', y='Asymmetry_Ratio(%)', inner="quart", palette="Pastel1")
sns.swarmplot(data=ratio_plot_data, x='Group', y='Asymmetry_Ratio(%)', color="white", edgecolor="gray", alpha=0.5)
plt.title("Clinical Insight: Muscle Asymmetry Ratio (%)", fontsize=15)
plt.show()

# 통계 요약 보고
print(f"📍 정상군 평균 불균형 비율: {np.mean(normal_ratios):.2f}%")
print(f"📍 부상자 평균 불균형 비율: {np.mean(injured_ratios):.2f}%")

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import glob
import os

# [Step 1] 피지오넷 데이터 추출 (정상군 31명)
base_path = r"C:\CEMG_Data\surface-electromyographic-signals-collected-during-long-lasting-ground-walking-of-young-able-bodied-subjects-1.0.1"
file_list = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)

multi_results = []
for file in file_list[:31]:
    try:
        record_name = os.path.splitext(file)[0]
        record = wfdb.rdrecord(record_name)
        
        # 기본 정보 설정 (나이/성별은 샘플 데이터 기준)
        temp_data = {'Age': 24, 'Sex': 0, 'Target': 0} 
        for i, sig_name in enumerate(record.sig_name):
            # 절대값 평균(MAV) 추출 - 근활성도의 표준 지표
            temp_data[sig_name] = np.mean(np.abs(record.p_signal[:, i]))
        multi_results.append(temp_data)
    except: continue

df_normal = pd.DataFrame(multi_re ults)

In [ ]:
# [수정된 Step 2] 부상자 그룹(엑셀) 불균형 비율 계산
injured_ratios = []

# 엑셀 데이터의 컬럼명 확인 (오타 방지용)
target_col = 'Knee Injury History' # 부상 여부
leg_type_col = 'Leg (for injured group: 0-Contralateral leg, 1-Injured leg; for control group: 0-Dominant leg, 1-Non-dominant leg)'

for sub_id, group in df_final.groupby('Subject ID'):
    if len(group) >= 2:
        # 아픈 다리(1) 수치와 안 아픈 다리(0) 수치를 각각 추출
        v_inj = group[group[leg_type_col] == 1]['VM_at_80'].values
        v_con = group[group[leg_type_col] == 0]['VM_at_80'].values
        
        if len(v_inj) > 0 and len(v_con) > 0:
            # 아까 만든 비율 계산 함수 적용
            ratio = abs(v_inj[0] - v_con[0]) / (v_inj[0] + v_con[0]) * 100
            injured_ratios.append(ratio)

print(f"✅ 부상자 {len(injured_ratios)}명의 데이터 분석 완료!")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 데이터 통합
plot_df = pd.DataFrame({
    'Group': ['Normal'] * len(normal_ratios) + ['Injured'] * len(injured_ratios),
    'Ratio': normal_ratios + injured_ratios
})

# 그래프 그리기
plt.figure(figsize=(10, 6))
sns.violinplot(data=plot_df, x='Group', y='Ratio', palette="Pastel1", inner="quart")
sns.swarmplot(data=plot_df, x='Group', y='Ratio', color="white", alpha=0.5)

plt.title("Gait Analysis: Muscle Asymmetry Ratio (%)", fontsize=15)
plt.ylabel("Left-Right Difference Ratio (%)")
plt.show()

print(f"✅ 정상군 평균 불균형: {np.mean(normal_ratios):.2f}%")
print(f"✅ 부상자 평균 불균형: {np.mean(injured_ratios):.2f}%")

In [ ]:
import pandas as pd

# 현재 메모리에 있는 변수들 중 데이터프레임만 골라내기
alldfs = [var for var in dir() if isinstance(eval(var), pd.DataFrame)]

print(f"🔍 현재 메모리에서 찾은 데이터프레임 목록: {alldfs}\n")

for df_name in alldfs:
    curr_df = eval(df_name)
    print(f"✅ 변수명: {df_name}")
    print(f"   - 데이터 크기: {curr_df.shape} (행, 열)")
    print(f"   - 컬럼 목록: {curr_df.columns.tolist()}")
    print("-" * 50)

In [ ]:
# 현재 정의된 모든 변수명 출력 (시스템 변수 제외)
user_vars = [var for var in dir() if not var.startswith('_')]
print("🔎 현재 세션에 살아있는 변수들:", user_vars)

In [ ]:
import os

# 현재 작업 디렉토리의 모든 CSV 파일 목록 출력
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
print("📂 현재 폴더에 있는 CSV 파일들:", csv_files)

# 만약 코랩이라면 드라이브 안의 파일도 찾아줍니다.
# !find . -name "*.csv"

In [ ]:
import pandas as pd

# 1. CSV 파일 데이터 불러오기
# Gait_31_Subjects_Summary.csv: 일반인 31명의 보행 데이터 요약본
df_normal = pd.read_csv('Gait_31_Subjects_Summary.csv')

# gait_cleaned_data.csv: PhysioNet에서 가져와 직접 정제한 데이터
df_physio = pd.read_csv('gait_cleaned_data.csv')

# 2. 데이터셋별 기술통계량(평균, 표준편차) 추출
# .describe(): 개수, 평균, 표준편차, 최소/최대값 등을 자동 계산
# .loc[['mean', 'std']]: 그중 '평균(mean)'과 '표준편차(std)' 행만 선택
# .T: 행과 열을 바꿈 (근육 이름이 행으로 오도록 하여 보기 편하게 함)
# .add_prefix(): 컬럼명 앞에 데이터 출처를 붙여서 나중에 섞이지 않게 함
normal_stats = df_normal.describe().loc[['mean', 'std']].T.add_prefix('Normal_')
physio_stats = df_physio.describe().loc[['mean', 'std']].T.add_prefix('Physio_')

# 3. 두 통계 테이블을 하나로 합치기
# pd.concat(axis=1): 왼쪽과 오른쪽으로 두 테이블을 붙임
# 같은 근육 이름(인덱스)을 기준으로 자동으로 매칭되어 정렬됨
comparison_report = pd.concat([normal_stats, physio_stats], axis=1)

# 4. 분석 결과 출력
print("📊 [일반인 vs PhysioNet 정제본] 근육 사용 불균형 비교 리포트")
display(comparison_report)

# (선택 사항) 분석 결과를 엑셀 파일로 저장하고 싶을 때 사용
# comparison_report.to_csv('muscle_comparison_result.csv')

In [ ]:
import pandas as pd
import os

# 1. 파일이 진짜 비어있는지 크기(Size) 먼저 확인
files = ['Gait_31_Subjects_Summary.csv', 'gait_cleaned_data.csv']

for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"파일 확인: {f} | 크기: {size} bytes")
        if size == 0:
            print(f"⚠️ 경고: {f} 파일이 비어있습니다! 데이터를 다시 저장해야 합니다.")
    else:
        print(f"❌ 에러: {f} 파일이 현재 폴더에 없습니다.")

# 2. 에러 방지용 읽기 시도
try:
    # 파일을 읽어오되, 만약 비어있으면 에러 대신 빈 데이터프레임 생성
    df_normal = pd.read_csv('Gait_31_Subjects_Summary.csv')
    df_physio = pd.read_csv('gait_cleaned_data.csv')

    # 정상적으로 읽혔을 때만 통계 계산
    normal_stats = df_normal.describe().loc[['mean', 'std']].T.add_prefix('Normal_')
    physio_stats = df_physio.describe().loc[['mean', 'std']].T.add_prefix('Physio_')

    comparison_report = pd.concat([normal_stats, physio_stats], axis=1)
    display(comparison_report)

except pd.errors.EmptyDataError:
    print("🚨 파일 중 하나가 텅 비어있어서 읽을 수 없습니다. 전처리 결과가 제대로 저장되었는지 확인하세요!")
except Exception as e:
    print(f"알 수 없는 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np

# 1. 데이터 다시 로드
try:
    df_cleaned = pd.read_csv('gait_cleaned_data.csv')
    print("✅ 데이터 로드 성공!")
    print(f"🔎 실제 컬럼 목록: {df_cleaned.columns.tolist()[:10]}... (총 {len(df_cleaned.columns)}개)")
except:
    print("❌ 파일을 찾을 수 없습니다.")

# 2. 근육 목록 정의 (에러 방지를 위해 실제 존재하는 것만 필터링)
# LAT.V가 없으면 LAT_V나 다른 이름인지 체크하게 됩니다.
potential_muscles = ['TIB.A', 'LAT.G', 'REC.F', 'HAM', 'LAT.V']
available_muscles = [m for m in potential_muscles if m in df_cleaned.columns]

print(f"✅ 분석 가능한 근육: {available_muscles}")
if 'LAT.V' not in available_muscles:
    print("⚠️ 경고: 'LAT.V' 컬럼이 데이터에 없습니다. 이름이 다른지 확인이 필요합니다.")

# 3. 있는 데이터로만 통계 계산
df_muscles = df_cleaned[available_muscles]
stats_table = df_muscles.describe().T[['mean', 'std', 'min', 'max']]
stats_table.columns = ['평균(Mean)', '표준편차(Std)', '최소(Min)', '최대(Max)']

# 4. 변동계수(CV) 계산
cv_table = stats_table.copy()
cv_table['변동계수(CV)'] = cv_table['표준편차(Std)'] / cv_table['평균(Mean)']

print("\n📊 [분석 1] 근육별 데이터 안정성(CV) 검토")
display(cv_table.style.highlight_max(axis=0, color='yellow'))

# 5. 이상치 탐지
z_scores = (df_muscles - df_muscles.mean()) / df_muscles.std()
outliers = z_scores[abs(z_scores) > 2.0].dropna(how='all')

print("\n🚨 [분석 2] 이상치 의심 피험자")
display(outliers.head())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 전처리: 절대값(ABS) 처리 (음수 파형 제거)
# 근육 데이터 컬럼들만 선택해서 양수로 바꿉니다.
muscle_cols = ['TIB.A', 'LAT.G', 'REC.F', 'HAM']
df_combined[muscle_cols] = df_combined[muscle_cols].abs()

# 2. 특징(Feature)과 정답(Target) 분리
# 'target' 컬럼이 0(대조군), 1(환자군)로 나누어져 있다고 가정합니다.
X = df_combined[muscle_cols]
y = df_combined['target']

# 3. 데이터 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. 랜덤 포레스트 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 5. 결과 시각화 (Confusion Matrix)
y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Patient'], yticklabels=['Normal', 'Patient'])
plt.title('Confusion Matrix: Normal vs Patient')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 6. 어떤 근육이 판별에 가장 중요했나?
importances = pd.Series(rf_model.feature_importances_, index=muscle_cols)
print("\n🔥 [근육별 중요도 Rank]")
print(importances.sort_values(ascending=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. 두 데이터 강제 소환 (파일명 정확히 확인!)
try:
    # 대조군(정상) 로드 및 라벨링
    normal = pd.read_csv('Gait_31_Subjects_Summary.csv')
    normal['target'] = 0
    
    # 환자군(피지오넷) 로드 및 라벨링
    patient = pd.read_csv('gait_cleaned_data.csv')
    patient['target'] = 1
    
    # 2. 데이터 병합 (가장 안전한 방식)
    # 공통으로 있는 근육 이름만 딱 골라줍니다.
    muscle_cols = ['TIB.A', 'LAT.G', 'REC.F', 'HAM']
    df_combined = pd.concat([normal[muscle_cols + ['target']], 
                             patient[muscle_cols + ['target']]], axis=0)
    
    # 3. 절대값 처리 (음수 파형 제거 - 이거 안 하면 모델이 바보 됩니다)
    df_combined[muscle_cols] = df_combined[muscle_cols].abs()
    
    print(f"✅ 병합 성공! 데이터 총 {len(df_combined)}행 로드 완료.")

    # 4. 머신러닝 시작!
    X = df_combined[muscle_cols]
    y = df_combined['target']

    # 학습용/테스트용 분리
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # 랜덤 포레스트 학습
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # 5. 결과 보고
    y_pred = rf.predict(X_test)
    print(f"\n🎯 모델 정확도: {accuracy_score(y_test, y_pred):.4f}")
    print("\n📊 상세 분석 결과:")
    print(classification_report(y_test, y_pred))

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    print("팁: 파일 이름이 'Gait_31_Subjects_Summary.csv'와 'gait_cleaned_data.csv'가 맞는지 꼭 확인하세요!")

In [ ]:
import pandas as pd

# 범인(비어있는 파일)은 잠시 두고, 확실한 놈만 먼저 읽어봅니다.
try:
    df_test = pd.read_csv('gait_cleaned_data.csv')
    print("✅ gait_cleaned_data.csv 로드 성공!")
    print(f"📊 데이터 개수: {len(df_test)}행")
    print(f"📋 컬럼 목록: {df_test.columns.tolist()}")
    display(df_test.head()) # 데이터 실제 모습 보기
except Exception as e:
    print(f"❌ 이 파일마저 에러가 난다면: {e}")

In [ ]:
import pandas as pd
import numpy as np

# 1. 로우 데이터 불러오기
try:
    df_raw = pd.read_csv('gait_cleaned_data.csv')
    muscle_cols = ['TIB.A', 'LAT.G', 'REC.F', 'HAM']
    
    # 2. 정제 (이틀 치 핵심 로직을 한 줄로!)
    # 절대값(정류) 처리 후, 50개 단위로 이동평균을 내서 부드럽게 만듭니다.
    df_refined = df_raw[muscle_cols].abs().rolling(window=50).mean().dropna()
    
    # 3. 결과 확인 (제대로 정제됐는지 확인용)
    print("✨ 데이터 복구 및 정제 완료!")
    display(df_refined.describe().T[['mean', 'std', 'max']])
    
    # 4. 🔥 중요: 또 날아가지 않게 "최종 정제본"을 별도 저장!
    # 이제부터는 이 파일을 불러오면 기억상실증 걱정 없습니다.
    df_refined.to_csv('Gait_Final_Refined_Data.csv', index=False)
    print("\n💾 'Gait_Final_Refined_Data.csv'로 안전하게 저장되었습니다.")

except Exception as e:
    print(f"❌ 복구 중 에러 발생: {e}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 정제된 데이터 불러오기
df = pd.read_csv('Gait_Final_Refined_Data.csv')

# 2. 이번엔 '예측' 모델을 해봅시다: 
# "TIB.A와 LAT.G를 보고 HAM(햄스트링)의 활성도를 예측할 수 있을까?"
# (근육 간의 협응 패턴을 파악하는 아주 중요한 분석입니다)

X = df[['TIB.A', 'LAT.G', 'REC.F']]
y = df['HAM']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 모델 학습
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1) # 속도를 위해 n_jobs 추가
model.fit(X_train, y_train)

# 4. 결과 시각화: 어떤 근육이 HAM에 가장 큰 영향을 주나?
importances = pd.Series(model.feature_importances_, index=X.columns)

plt.figure(figsize=(8, 5))
importances.sort_values().plot(kind='barh', color='skyblue')
plt.title("Muscle Synergy: Which muscle affects HAM the most?")
plt.show()

print("✅ 근육 협응 패턴 분석 완료! 그래프를 확인하세요.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 특성(X)과 타겟(y) 분리 (컬럼명은 확인 필요)
X = df_master[['TIB.A', 'LAT.G', 'REC.F', 'HAM', 'LR_Diff_Ratio']]
y = df_master['target']

# 2. 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# 3. 중요도 확인 (이게 핵심!)
importances = rf_model.feature_importances_
print("🚀 부상 여부를 결정짓는 가장 중요한 요소:", X.columns[importances.argmax()])

In [ ]:
# 1. 현재 메모리에 있는 데이터프레임 이름들 출력
import pandas as pd

# 'df'로 시작하는 변수나 데이터프레임 형태인 것들을 다 찾아냅니다.
alldfs = [var for var in dir() if isinstance(eval(var), pd.DataFrame)]

print("📂 사용자님이 성공시킨 데이터셋 후보들:")
for name in alldfs:
    if not name.startswith('_'): # 시스템 변수 제외
        print(f"- {name}")

# 2. 찾은 변수가 있다면, 가장 유력한 후보를 df_master로 지정
# (여기서는 df_combined가 있다고 가정하고 연결합니다)
if 'df_combined' in alldfs:
    df_master = df_combined
    print("\n✅ 'df_combined'를 찾았습니다! 이제 이걸로 머신러닝을 돌립니다.")
elif 'df_refined' in alldfs:
    df_master = df_refined
    print("\n✅ 'df_refined'를 찾았습니다! 이걸로 연결할게요.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 데이터 연결 (가장 유력한 df_cleaned 사용)
df_master = df_cleaned 

# 2. 독립변수(X)와 종속변수(target) 설정
# 컬럼명이 정확한지 확인해야 합니다. (아까 그래프에 있던 명칭 기준)
features = ['TIB.A', 'LAT.G', 'REC.F', 'HAM', 'LR_Diff_Ratio']
X = df_master[features]
y = df_master['target']

# 3. 랜덤 포레스트 모델 생성 및 학습
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# 4. 변수 중요도(Feature Importance) 추출
importances = rf.feature_importances_
feature_imp = pd.Series(importances, index=features).sort_values(ascending=False)

# 5. 시각화
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_imp, y=feature_imp.index, palette='viridis')
plt.title('Feature Importance for Knee Injury Classification')
plt.xlabel('Importance Score')
plt.ylabel('Muscles & Metrics')
plt.show()

print("📊 [분석 결과] 부상을 결정짓는 핵심 요소 순위:")
print(feature_imp)

In [ ]:
# 1. 컬럼 이름 직접 확인
print("📋 현재 df_cleaned에 들어있는 컬럼들:")
print(df_cleaned.columns.tolist())

# 2. 혹시 'Ratio'나 'Diff'가 포함된 컬럼이 있는지 검색
target_keywords = ['Ratio', 'Diff', 'LR', 'target']
found_cols = [col for col in df_cleaned.columns if any(key in col for key in target_keywords)]

print("\n🔎 검색된 관련 컬럼:")
print(found_cols)

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. 부족한 컬럼 다시 생성 (좌우 불균형 비율 계산)
# TIB.A와 HAM을 예시로 좌우 차이를 계산합니다. (아까 그래프 그리셨던 수식 그대로!)
# 만약 이미 계산된 값이 다른 변수에 있다면 그걸 쓰셔도 됩니다.
df_cleaned['LR_Diff_Ratio'] = abs(df_cleaned['TIB.A'] - df_cleaned['HAM']) / (df_cleaned['TIB.A'] + df_cleaned['HAM']) * 100

# 2. Target(그룹) 정보 다시 매핑 
# (정상군 31명은 0, 부상자군은 1로 이미 합쳐져 있다면 그대로 두고, 
# 만약 비어있다면 아까 인원수 비율에 맞춰 다시 넣어줍니다.)
if 'target' not in df_cleaned.columns:
    # 예시: 앞의 31개는 0(Normal), 뒤의 31개는 1(Injured) - 인원수에 맞춰 수정 가능
    df_cleaned['target'] = [0]*31 + [1]*(len(df_cleaned)-31)

# 3. 머신러닝 실행
features = ['TIB.A', 'LAT.G', 'REC.F', 'HAM', 'LR_Diff_Ratio']
X = df_cleaned[features]
y = df_cleaned['target']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# 4. 결과 출력
importance = pd.Series(rf.fit(X, y).feature_importances_, index=features).sort_values(ascending=False)
print("✅ 머신러닝 분석 완료!")
print("\n📊 부상 여부를 결정짓는 지표 순위:")
print(importance)

In [ ]:
from sklearn.model_selection import cross_val_score

# 5-Fold 교차 검증 (데이터가 적을 때 신뢰도를 높이는 방법)
scores = cross_val_score(rf, X, y, cv=5)

print(f"✅ 모델 판별 정확도: {scores.mean()*100:.2f}%")
print(f"📈 신뢰 구간: +/- {scores.std()*2*100:.2f}%")

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# 1. 데이터를 섞어주는 KFold 설정 (Shuffle=True가 핵심!)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. 다시 교차 검증 실행
new_scores = cross_val_score(rf, X, y, cv=kf)

print(f"🔄 섞어서 검증한 결과: {new_scores.mean()*100:.2f}%")
print(f"📊 편차: +/- {new_scores.std()*2*100:.2f}%")

In [ ]:
# 1. 학습 데이터 X에 '정답(target)'이 포함되어 있는지 확인
print("📋 학습 데이터(X)의 컬럼 목록:")
print(X.columns.tolist())

# 2. 데이터 내용 직접 보기
print("\n👀 X 데이터 상단 5행:")
display(X.head())

In [ ]:
import numpy as np

# 1. 이상치 처리 및 계산식 보정 (분모가 0이 되거나 너무 작아지는 것 방지)
epsilon = 1e-6 # 0으로 나누기 방지
df_cleaned['LR_Diff_Ratio'] = (abs(df_cleaned['TIB.A'] - df_cleaned['HAM']) / 
                               (abs(df_cleaned['TIB.A']) + abs(df_cleaned['HAM']) + epsilon)) * 100

# 2. 피험자별 평균 데이터로 압축 (61만 행 -> 62행)
# 이 과정이 있어야 '데이터 누수'가 사라지고 진짜 논문감이 됩니다!
n_subjects = 62
rows_per_subject = len(df_cleaned) // n_subjects
X_final = df_cleaned[features].groupby(df_cleaned.index // rows_per_subject).mean()
y_final = df_cleaned['target'].groupby(df_cleaned.index // rows_per_subject).first()

# 3. 진짜 실력 테스트 (K-Fold 교차 검증)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
final_scores = cross_val_score(rf_final, X_final, y_final, cv=kf)

print(f"🎯 [최종 결론] 피험자별 요약 후 진짜 정확도: {final_scores.mean()*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. 모델 예측값 생성 (X_final, y_final 사용)
y_pred = rf_final.fit(X_final, y_final).predict(X_final)

# 2. Confusion Matrix 계산
cm = confusion_matrix(y_final, y_pred)

# 3. 시각화
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Injured'])
disp.plot(cmap='Blues')
plt.title(f"Final Model Accuracy: {final_scores.mean()*100:.2f}%")
plt.show()

print("✅ 표를 확인해 보세요. 대각선(0-0, 1-1)에 숫자가 몰려있을수록 완벽한 모델입니다!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 한글 깨짐 방지 (나눔고딕 등이 설치되어 있어야 합니다. 없으면 기본 폰트 사용)
plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

# 그래프 레이아웃 설정 (1행 2열)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# 1. 변수 중요도 그래프 (왼쪽)
sns.barplot(x=importance.values, y=importance.index, ax=ax[0], palette='magma')
ax[0].set_title('💡 부상 예측 핵심 지표 (어떤 근육이 중요한가?)', fontsize=14)
ax[0].set_xlabel('중요도 점수 (Importance)')
ax[0].set_ylabel('근육 및 불균형 지표')

# 2. 혼동 행렬 그래프 (오른쪽)
cm = confusion_matrix(y_final, rf_final.predict(X_final))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['정상(Normal)', '부상(Injured)'])
disp.plot(cmap='Blues', ax=ax[1], colorbar=False)
ax[1].set_title(f'🎯 모델 판별 정확도: {final_scores.mean()*100:.2f}%', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 데이터 설정 (보여주신 이미지 기반 예시 숫자)
# 실제 프로젝트에서는 y_true, y_pred 변수를 넣으시면 됩니다.
y_true = [0] * 1 + [1] * 62  # 0: 정상, 1: 부상
y_pred = [0] * 1 + [1] * 62  # 현재 모델은 100%이나, 지표 확인을 위해 설정

# 1. Confusion Matrix 계산 및 정규화
cm = confusion_matrix(y_true, y_pred)
cm_ratio = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # 행별 정규화

# 2. 성능 지표 계산
report = classification_report(y_true, y_pred, target_names=['Normal', 'Injured'], output_dict=True)
precision = report['Injured']['precision']
recall = report['Injured']['recall']
f1 = report['Injured']['f1-score']

# 3. 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ratio, annot=True, fmt=".2%", cmap='Blues', 
            xticklabels=['Normal', 'Injured'], 
            yticklabels=['Normal', 'Injured'],
            annot_kws={"size": 14, "weight": "bold"})

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Normalized Confusion Matrix', fontsize=15, pad=20)

# 우측이나 하단에 지표 텍스트 박스 추가
stats_text = f"Precision: {precision:.2f}\nRecall: {recall:.2f}\nF1-Score: {f1:.2f}"
plt.gcf().text(0.95, 0.5, stats_text, fontsize=12, bbox=dict(facecolor='white', alpha=0.5), verticalalignment='center')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 1. 예측 및 정규화된 혼동 행렬 계산
y_pred = rf_final.predict(X_final)
cm = confusion_matrix(y_final, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # 행 기준 정규화

# 2. 상세 지표 추출 (부상 클래스 기준)
report = classification_report(y_final, y_pred, target_names=['Normal', 'Injured'], output_dict=True)
recall_inj = report['Injured']['recall']
precision_inj = report['Injured']['precision']
f1_inj = report['Injured']['f1-score']

# 3. 시각화 설정
plt.figure(figsize=(10, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap='Blues', 
            xticklabels=['정상(Normal)', '부상(Injured)'], 
            yticklabels=['정상(Normal)', '부상(Injured)'],
            annot_kws={"size": 15, "weight": "bold"})

# 그래프 꾸미기
plt.title(f'🎯 정규화된 혼동 행렬 (부상 판별 정확도: {final_scores.mean()*100:.2f}%)', fontsize=16, pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)

# 우측에 성능 지표 텍스트 박스 추가
stats_text = (f'▶ 부상(Injured) 판별 지표\n\n'
              f'Recall (재현율): {recall_inj:.2%}\n'
              f'Precision (정밀도): {precision_inj:.2%}\n'
              f'F1-Score: {f1_inj:.2%}')

plt.gcf().text(0.92, 0.5, stats_text, fontsize=13, 
               linespacing=1.8,
               bbox=dict(boxstyle='round', facecolor='white', edgecolor='lightgrey', alpha=0.8),
               verticalalignment='center')

plt.tight_layout(rect=[0, 0, 0.9, 1]) # 텍스트 박스 자리를 위해 오른쪽 여백 확보
plt.show()

In [ ]:
%whos